# JAX DR Bridge Estimator — Gamma Surface Sweep

GPU/TPU/CPU-accelerated Doubly Robust Bridge Estimator for the Navegador gamma-surface.

- **float64** precision (with `jax_enable_x64`)
- **`jnp.linalg.solve`** for Newton steps
- **vmap** over bootstrap iterations for massive parallelism

**Workflow:** Setup → Clone → Run sweep (notebook cells or CLI) → Download results.

The compute code lives in standalone Python files under `scripts/`:
- `scripts/jax_dr_bridge.py` — JAX estimator module (all pure functions)
- `scripts/load_pairs.py` — data loading + sweep task builder
- `scripts/run_sweep.py` — batched sweep runner + results summary

These can be run from the notebook (below) or directly from terminal:
```bash
cd /content/navegador
python scripts/run_sweep.py --dataset wvs_temporal --data-repo /content/navegador_data
```

## Cell 1: Setup — Claude Code CLI + JAX

In [ ]:
# --- Claude Code CLI ---
!npm install -g @anthropic-ai/claude-code 2>&1 | tail -3

import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('ANTHROPIC_API_KEY set from Colab Secrets')
except Exception:
    print('No API key in Colab Secrets — run !claude to authenticate interactively')

# --- JAX setup ---
os.environ['JAX_ENABLE_X64'] = '1'

import jax
jax.config.update('jax_enable_x64', True)
backend = jax.default_backend()
print(f'Backend: {backend}, Devices: {jax.devices()}')

import jax.numpy as jnp
print(f'Float64: {jnp.array(1.0).dtype}')

## Cell 2: Clone Repos + Set Dataset

Clones the `navegador` repo (branch `colab_jax`, for code + scripts) and `navegador_data` repo (for bridge CSVs + sweep results).

Set `DATASET` to choose which sweep to run:
- `"los_mex"` — 24 domain CSVs, ~4,100 construct pairs (cross-domain)
- `"wvs"` — 66 country CSVs (Wave 7), ~68,000 pairs (cross-country)
- `"wvs_temporal"` — 72 contexts: 7 Mexico waves (W1-W7) + 66 W7 countries, ~86,000 pairs

Both repos are public — no authentication needed.

In [ ]:
import os

DATASET = "wvs_temporal"  # "los_mex" | "wvs" | "wvs_temporal"
REPO_DIR = "/content/navegador"
DATA_REPO_DIR = "/content/navegador_data"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch colab_jax https://github.com/salvadorVMA/navegador.git {REPO_DIR}
else:
    print(f"navegador already cloned at {REPO_DIR}")

if not os.path.exists(DATA_REPO_DIR):
    !git clone --depth 1 https://github.com/salvadorVMA/navegador_data.git {DATA_REPO_DIR}
else:
    print(f"navegador_data already cloned at {DATA_REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"CLAUDE.md exists: {os.path.exists('CLAUDE.md')}")

## Cell 3: Load Data + Build Pairs

Reads the manifest, loads CSVs, and builds the sweep task list.

Equivalent CLI:
```bash
python scripts/load_pairs.py --dataset wvs_temporal --data-repo /content/navegador_data
```

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from scripts.load_pairs import load_sweep_tasks

context_dfs, sweep_tasks, config = load_sweep_tasks(DATASET, DATA_REPO_DIR)

## Cell 4: Run Sweep

Runs all pairs with batched checkpointing. Resume-safe — re-run to continue from last checkpoint.

**CLI alternative** (run from terminal, e.g. on a TPU VM):
```bash
cd /content/navegador
python scripts/run_sweep.py --dataset wvs_temporal --data-repo /content/navegador_data \
    --n-sim 2000 --n-bootstrap 200 --batch-size 200
```

In [ ]:
from scripts.run_sweep import run_sweep

results = run_sweep(
    context_dfs, sweep_tasks, config,
    n_sim=2000,
    n_bootstrap=200,
    batch_size=200,
)

## Cell 5: Results Summary + Download

Prints summary statistics and downloads the results JSON.

CLI equivalent:
```bash
python -c "
import json; from scripts.run_sweep import print_summary
print_summary(json.load(open('/content/jax_dr_sweep_wvs_temporal.json')))
"
```

In [ ]:
from scripts.run_sweep import print_summary

print_summary(results)

# Download results JSON
try:
    from google.colab import files
    output_file = f'/content/jax_dr_sweep_{DATASET}.json'
    files.download(output_file)
except ImportError:
    print("Not in Colab — results saved to disk.")